First, let's import the required packages for embedding and vector storage.

In [5]:
# Import embedding and vector database libraries
from sentence_transformers import SentenceTransformer
import chromadb
import os
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from chromadb.utils import embedding_functions

res_folder = "nls-data/"

# Configuration for vector database
db_collection_name = "nls-data-foundry"  # Name for our vector collection
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"  # Pre-trained embedding model
path_chroma = os.path.join(res_folder, "chroma_db")  # Path to store the vector database
input_content_dir = os.path.join(res_folder, "nls-sample")  # Directory with cleaned text files
#input_url_dir = os.path.join(res_folder, "covid19_corpus/cleaned/url")  # Directory with cleaned text files
#GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Qwen model on Hugging Face

print(path_chroma)

nls-data/chroma_db


ChromaDB is a vector database designed for efficient storage and retrieval of embeddings. It allows us to perform similarity searches across our corpus.

In [7]:
# Initialize ChromaDB client and collection
client = chromadb.PersistentClient(path=path_chroma)

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
     model_name=embedding_model
)
#embedding_model = HuggingFaceEmbeddings(model_name=GEN_MODEL)


collection = client.get_or_create_collection(name=db_collection_name,
                                             embedding_function=embedding_func)

# Load the sentence transformer model for creating embeddings
model = SentenceTransformer(embedding_model, token=False)

# Get all cleaned text files
files = sorted([f for f in os.listdir(input_content_dir) if f.endswith(".txt")])

# Process each file and add its contents to the vector database
for fname in files:
    # Load content file
    content_file_path = os.path.join(input_content_dir, fname)
    with open(content_file_path, encoding="utf-8") as f:
        lines = [line.strip() for line in f]
    
    # Load corresponding URL file
    #url_file_path = os.path.join(input_url_dir, fname.replace("content", "url"))
    #with open(url_file_path, encoding="utf-8") as f:
    #    urls = [line.strip() for line in f]
    
    # Create embeddings for each line of text
    # This converts text into numerical vectors that capture semantic meaning
    embeddings = model.encode(lines, show_progress_bar=True, convert_to_numpy=True)
    #datestr = extract_date(fname)
    
    # Add embeddings and metadata to ChromaDB
    try:
        collection.add(
            ids=[f"{fname}_{i}" for i in range(len(lines))],
            embeddings=embeddings.tolist(),
            documents=lines,
            #metadatas=[{"filename": fname, "date": datestr, "line": i, "url": url} for i, url in enumerate(urls)]
            metadatas=[{"filename": fname}]
        )
    except ValueError:
        print(f"⚠️ Skipped {fname} due to ValueError")

print("✅ Indexed all lines into Chroma!")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed all lines into Chroma!


In [8]:
def semantic_search(query, n_results=2):
    """Perform semantic search on the corpus using vector embeddings.
    
    Args:
        query: The search query as a string
        n_results: Number of results to return (default: 5)
        
    Returns:
        Dictionary containing search results from ChromaDB
    """
    # Load the embedding model
    model = SentenceTransformer(embedding_model)

    # Connect to the vector database
    client = chromadb.PersistentClient(path=path_chroma)
    collection = client.get_or_create_collection(name=db_collection_name)

    # Convert query to embedding vector
    query_emb = model.encode([query], convert_to_numpy=True).tolist()

    # Search for similar content in the database
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results
    )

    # Display results with metadata
    for text, meta in zip(results["documents"][0], results["metadatas"][0]):
        #print(f"{meta['filename']} (line {meta['line']}) — {meta['url']}")
        print(f"{meta['filename']}")
        print(f"→ {text[:]}\n")

    return results

In [11]:
# Example semantic search query
# This demonstrates finding content related to economic impacts without requiring exact keyword matches
#res = semantic_search("Which hospitals treated leprosy?")
res = semantic_search("infected villages?", 2)

74457530.txt
→ From the above it will be seen that the highest rate to population is found in Khandeish, and next to it in Ahmednagar, whilst in the Puna, Sholapur, Satara and Dharwar Districts which are situated above the Ghats, and in the Ratnagiri and Kolaba Collectorates on the sea coast, the ratio to population is high. In the other districts the disease appears to be proportionally less, the least number being returned from the Province of Sind. I quite concur in the opinion expressed by the Government of India that any measures of segragation and medical treatment of lepers throughout the country would be impracticable as a State measure, but I do hold that the improvement of the hygienic conditions under which the mass of the people live is the only sure method of stamping out leprosy or any similar disease. The filth with which all the villages in India are surrounded is quite sufficient to prevent any hope of success in combating the disease, which it is not difficult to fore